# Fase 3: Data Preparation (Consolidación Final)
En esta sección unimos los datos de Aeropuertos (LIR/SJO), Ocupación (BCCR), y Variables Macro (FRED + yfinance). 

**Nota:** Hemos incluido un proceso de 'Data Infill' para corregir los vacíos (NaN) de finales de 2025 y inicios de 2026 usando datos rescatados de fuentes oficiales (BLS y EIA).

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

print('Librerías listas.')

### 1. Ingesta de Datos Macroeconómicos
Combinamos FRED (Seguimiento Histórico) con `yfinance` (Datos Real-Time).

In [ ]:
# A. Descarga desde FRED (Fondo Histórico)
start_date = '2009-01-01'
series = {
    'UNRATE': 'tasa_desempleo_usa',                 
    'CPIAUCSL': 'inflacion_usa_cpi'
}

df_list = []
for s_id, s_name in series.items():
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={s_id}'
    df_t = pd.read_csv(url, parse_dates=['observation_date'], na_values='.')
    df_t = df_t[df_t['observation_date'] >= start_date]
    df_t.set_index('observation_date', inplace=True)
    df_t = df_t.resample('ME').mean().rename(columns={s_id: s_name})
    df_list.append(df_t)

df_macro = pd.concat(df_list, axis=1).reset_index()
df_macro.rename(columns={'observation_date': 'DATE'}, inplace=True)

# B. Descarga desde yfinance (Petróleo WTI Real-Time)
print('Descargando Petróleo desde Yahoo Finance...')
oil = yf.download('CL=F', start=start_date, interval='1mo')

# Ajuste robusto para yfinance MultiIndex
if isinstance(oil.columns, pd.MultiIndex):
    # Si es MultiIndex, buscamos la columna Close del ticker
    oil_df = oil.xs('Close', axis=1, level=0)
else:
    oil_df = oil[['Close']]

oil_df = oil_df.resample('ME').mean()
oil_df.columns = ['precio_petroleo_wti']
oil_df.index.name = 'DATE'
oil_df = oil_df.reset_index()

df_macro = pd.merge(df_macro, oil_df, on='DATE', how='left')
print('Ingesta completa.')

### 2. Manual Data Infill (Parchado de Datos Recientes)
Rellenamos los huecos de 2025-2026 con los datos rescatados del BLS para evitar `NaN` en el modelo.

In [ ]:
# Diccionario de Datos Rescatados (BLS 2025-2026)
PATCH_DATA = {
    '2026-01-31': {'tasa_desempleo_usa': 4.3, 'inflacion_usa_cpi': 250.5, 'precio_petroleo_wti': 60.04},
    '2026-02-28': {'tasa_desempleo_usa': 4.4, 'inflacion_usa_cpi': 251.2, 'precio_petroleo_wti': 64.51},
    '2026-03-31': {'tasa_desempleo_usa': 4.3, 'inflacion_usa_cpi': 252.8, 'precio_petroleo_wti': 91.38} 
}

# Aplicar el parche
for dt_str, values in PATCH_DATA.items():
    ts = pd.to_datetime(dt_str)
    if ts in df_macro['DATE'].values:
        idx = df_macro[df_macro['DATE'] == ts].index[0]
        for col, val in values.items():
            if pd.isna(df_macro.loc[idx, col]):
                df_macro.loc[idx, col] = val

print('✅ Gaps de 2026 parchados con éxito.')

### 3. Fusión Maestra (Merge Final)
Unimos Llegadas (SJO + LIR) + Ocupación + Macro en el dataset definitivo.

In [ ]:
# Cargar datos de turismo procesados
df_arr = pd.read_csv('../data/processed/arrivals_clean.csv')
df_arr['DATE'] = pd.to_datetime(df_arr[['Year', 'Month']].assign(day=1)) + pd.offsets.MonthEnd(0)

df_occ = pd.read_csv('../data/processed/occupancy_clean.csv')
df_occ['DATE'] = pd.to_datetime(df_occ[['Year', 'Month']].assign(day=1)) + pd.offsets.MonthEnd(0)

# Fusionar todo
df_master = pd.merge(df_arr, df_occ[['DATE', 'Guanacaste_Occupancy_Pct']], on='DATE', how='left')
df_master = pd.merge(df_master, df_macro, on='DATE', how='left')

os.makedirs('../data/merged', exist_ok=True)
df_master.to_csv('../data/merged/master_tourism_data.csv', index=False)

print(f"¡Merge Finalizado! Dataset listo para el modelo: {df_master.shape[0]} meses.")
display(df_master.tail())

### 4. Visualización del Doble Motor (SJO vs LIR)
Analizamos cómo se comportan los dos aeropuertos respecto a la ocupación.

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.set_xlabel('Año')
ax1.set_ylabel('Llegadas Internacionales', color='blue')
ax1.plot(df_master['DATE'], df_master['Arrivals_sjo'], label='Arrivals SJO', color='lightblue', alpha=0.7)
ax1.plot(df_master['DATE'], df_master['Arrivals_lir'], label='Arrivals LIR', color='darkblue')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.set_ylabel('Ocupación Guanacaste (%)', color='red')
ax2.plot(df_master['DATE'], df_master['Guanacaste_Occupancy_Pct'], label='Ocupación (%)', color='red', linewidth=3, linestyle='--')
ax2.tick_params(axis='y', labelcolor='red')

plt.title('Doble Motor (SJO + LIR) vs Ocupación Hotelera')
fig.tight_layout()
plt.show()

### 5. Matriz de Correlación Final
Verificamos la fuerza de los indicadores.

In [ ]:
plt.figure(figsize=(10, 6))
cols = ['Arrivals_sjo', 'Arrivals_lir', 'Guanacaste_Occupancy_Pct', 'tasa_desempleo_usa', 'precio_petroleo_wti']
sns.heatmap(df_master[cols].corr(), annot=True, cmap='RdYlGn', center=0)
plt.show()